# 11 — Logical-access front-ends and AASIST t-DCF

Nothing is trained. The notebook scores checkpoints that are already on disk.

1. Mixed log-Mel and mixed LFCC on the official ASVspoof 2019 logical-access evaluation list. The mixed inverted-Mel score, 49.33%, is left as it is.
2. Published AASIST weights on the physical-access evaluation list only. The 2017 AASIST run is not repeated. This pass writes a per-file score file and then the minimum t-DCF.

A finished score file is not written again if you restart the notebook. The AASIST physical-access pass is the long job. Do not start another GPU job while this is running.


In [2]:
import json
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "common.py").exists():
    HERE = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval")
sys.path.insert(0, str(HERE))

from common import experiments_dir, paper_runs, protocol_count, run
from compute_pa_tdcf import min_tdcf_for_predictions

EXPERIMENTS = experiments_dir()
RUNS = paper_runs()
LA_COUNT = protocol_count("la_eval")
PA_COUNT = protocol_count("pa_eval")

CHECKPOINTS = {
    "mixed log-Mel": EXPERIMENTS / "lfcc_vs_mel_compare" / "runs" / "mel" / "best_mel_mixed_2017_pa2019.pt",
    "mixed LFCC": EXPERIMENTS / "lfcc_vs_mel_compare" / "runs" / "lfcc" / "best_lfcc_mixed_2017_pa2019.pt",
}
LA_OUTPUT = {
    "mixed log-Mel": RUNS / "mixed_mel_on_la_eval",
    "mixed LFCC": RUNS / "mixed_lfcc_on_la_eval",
}
AASIST = EXPERIMENTS / "aasist_zeroshot" / "AASIST.pth"
AASIST_PREDICTIONS = RUNS / "aasist_pa_eval" / "pa2019_eval_predictions.csv"
SUMMARY = RUNS / "la_frontends_and_aasist_tdcf.json"

missing = [path for path in [*CHECKPOINTS.values(), AASIST] if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing checkpoint: " + ", ".join(str(path) for path in missing))


def finished_metrics(path: Path, expected: int) -> dict | None:
    metrics_path = path / "metrics.json"
    if not metrics_path.is_file():
        return None
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    scored = int(metrics.get("num_scored_files", -1))
    skipped = int(metrics.get("num_skipped_corrupt", 0))
    if scored == expected and skipped == 0:
        return metrics
    return None


def csv_data_rows(path: Path) -> int:
    if not path.is_file():
        return 0
    with path.open(encoding="utf-8") as handle:
        return max(sum(1 for line in handle if line.strip()) - 1, 0)


la_rows = []
for name, checkpoint in CHECKPOINTS.items():
    output = LA_OUTPUT[name]
    metrics = finished_metrics(output, LA_COUNT)
    if metrics is None:
        run([
            sys.executable,
            str(HERE / "score_la_eval.py"),
            "--split", "eval",
            "--batch-size", "16",
            "--checkpoint", str(checkpoint),
            "--output", str(output),
        ])
        metrics = finished_metrics(output, LA_COUNT)
        if metrics is None:
            raise RuntimeError(f"{name} did not score all {LA_COUNT} LA eval files")
    else:
        print(f"skip {name}: {output / 'metrics.json'}")
    la_rows.append({
        "model": name,
        "checkpoint": str(checkpoint.resolve()),
        "eer_percent": metrics["eer_percent"],
        "num_scored_files": metrics["num_scored_files"],
        "metrics": str((output / "metrics.json").resolve()),
    })

if csv_data_rows(AASIST_PREDICTIONS) != PA_COUNT:
    run([
        sys.executable,
        str(HERE / "score_aasist_official_eval.py"),
        "--corpus", "pa",
        "--batch-size", "8",
    ])
else:
    print(f"skip AASIST PA: {AASIST_PREDICTIONS}")

if csv_data_rows(AASIST_PREDICTIONS) != PA_COUNT:
    raise RuntimeError(f"AASIST did not write all {PA_COUNT} PA eval scores")

aasist_metrics = json.loads((RUNS / "aasist_pa_eval" / "metrics.json").read_text(encoding="utf-8"))
min_tdcf = min_tdcf_for_predictions(AASIST_PREDICTIONS)
summary = {
    "la_eval": la_rows,
    "aasist_pa": {
        "checkpoint": str(AASIST.resolve()),
        "eer_percent": aasist_metrics["eer_percent"],
        "num_scored_files": aasist_metrics["num_scored_files"],
        "spoof_class_index": 0,
        "min_tdcf": min_tdcf,
        "predictions": str(AASIST_PREDICTIONS.resolve()),
    },
}
SUMMARY.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"saved {SUMMARY}")


skip mixed log-Mel: D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\runs\mixed_mel_on_la_eval\metrics.json

$ c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe d:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\score_la_eval.py --split eval --batch-size 16 --checkpoint D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare\runs\lfcc\best_lfcc_mixed_2017_pa2019.pt --output D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\runs\mixed_lfcc_on_la_eval

$ c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe d:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval\score_aasist_official_eval.py --corpus pa --batch-size 8
{
  "la_eval": [
    {
      "model": "mixed log-Mel",
      "checkpoint": "D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\lfcc_vs_mel_compare\\runs\\mel\\best_mel_mixed_2017_pa